In [ ]:
import numpy as np
import pandas as pd

In [ ]:
rng = np.random.default_rng(42)
print("ready")

In [ ]:
n_employees = 1000

employees = pd.DataFrame({
    "UserPrincipalName": [
        f"user{i:04d}@nordlys-demo.example"
        for i in range(1, n_employees + 1)
    ],
    "HomeCity": rng.choice(
        ["Oslo", "Bergen", "Stavanger", "Stockholm"],
        size=n_employees,
        p=[0.6, 0.15, 0.15, 0.1],
    ),
})

city_to_country = {
    "Oslo": "NO",
    "Bergen": "NO",
    "Stavanger": "NO",
    "Stockholm": "SE",
}
employees["HomeCountry"] = employees["HomeCity"].map(city_to_country)

employees.head()

In [ ]:
days = pd.date_range("2026-09-01", periods=30, freq="D", tz="UTC")

base = employees.merge(pd.DataFrame({"Day": days}), how="cross")

is_weekend = base["Day"].dt.dayofweek >= 5
average_logins = np.where(is_weekend, 0.2, 2.5)
login_counts = rng.poisson(average_logins)

signins = base.loc[base.index.repeat(login_counts)].reset_index(drop=True)

print(len(signins))

In [ ]:
n_signins = len(signins)

seconds_of_day = rng.normal(7 * 3600, 2.5 * 3600, n_signins)
seconds_of_day = (seconds_of_day % 86400).astype(int)

signins["TimeGenerated"] = signins["Day"] + pd.to_timedelta(seconds_of_day, unit="s")

signins = signins.drop(columns="Day")
signins = signins.sort_values("TimeGenerated").reset_index(drop=True)

signins.head()

In [ ]:
office_ip = {
    "Oslo": "203.0.113.10",
    "Bergen": "203.0.113.20",
    "Stavanger": "203.0.113.30",
    "Stockholm": "203.0.113.40",
}

signins["IPAddress"] = signins["HomeCity"].map(office_ip)
signins["Location"] = signins["HomeCountry"]
signins["City"] = signins["HomeCity"]

signins["ResultType"] = rng.choice(
    [0, 50126],
    size=len(signins),
    p=[0.97, 0.03],
)

signins["AppDisplayName"] = rng.choice(
    ["Outlook", "Teams", "SharePoint"],
    size=len(signins),
    p=[0.5, 0.35, 0.15],
)

signins.head()

In [ ]:
spray_index = rng.choice(len(employees), size=150, replace=False)
spray = employees.iloc[spray_index].copy()

spray["TimeGenerated"] = pd.Timestamp("2026-09-08 02:14:00", tz="UTC") + pd.to_timedelta(
    rng.integers(0, 2400, size=150), unit="s"
)
spray["IPAddress"] = "198.51.100.77"
spray["Location"] = "NL"
spray["City"] = "Amsterdam"
spray["ResultType"] = 50126
spray["AppDisplayName"] = "Outlook"
spray["AttackType"] = "password_spraying"

print(len(spray))
spray.head()

In [ ]:
travel_base = signins[
    (signins["ResultType"] == 0) & (signins["HomeCountry"] == "NO")
].sample(n=5, random_state=7)

travel = travel_base[
    ["UserPrincipalName", "HomeCity", "HomeCountry", "TimeGenerated", "AppDisplayName"]
].copy()

travel["TimeGenerated"] = travel["TimeGenerated"] + pd.Timedelta(minutes=25)
travel["IPAddress"] = "198.51.100.120"
travel["Location"] = "SG"
travel["City"] = "Singapore"
travel["ResultType"] = 0
travel["AttackType"] = "impossible_travel"

print(len(travel))
travel.head()

In [ ]:
mfa_rows = []

for user_index in rng.choice(len(employees), size=3, replace=False):
    user = employees.iloc[user_index]
    start = pd.Timestamp("2026-09-15 19:30:00", tz="UTC") + pd.Timedelta(
        days=int(rng.integers(0, 10)),
        minutes=int(rng.integers(0, 90)),
    )
    n_prompts = int(rng.integers(12, 20))

    for i in range(n_prompts + 1):
        mfa_rows.append({
            "UserPrincipalName": user["UserPrincipalName"],
            "HomeCity": user["HomeCity"],
            "HomeCountry": user["HomeCountry"],
            "TimeGenerated": start + pd.Timedelta(seconds=30 * i),
            "IPAddress": "198.51.100.200",
            "Location": "DE",
            "City": "Frankfurt",
            "ResultType": 0 if i == n_prompts else 500121,
            "AppDisplayName": "Outlook",
            "AttackType": "mfa_fatigue",
        })

mfa = pd.DataFrame(mfa_rows)

print(len(mfa))
mfa.tail()

In [ ]:
compromised = pd.concat([
    travel[["UserPrincipalName", "TimeGenerated", "IPAddress"]],
    mfa[mfa["ResultType"] == 0][["UserPrincipalName", "TimeGenerated", "IPAddress"]],
]).sample(n=4, random_state=11)

inbox_attacks = pd.DataFrame({
    "TimeGenerated": compromised["TimeGenerated"] + pd.Timedelta(minutes=10),
    "UserId": compromised["UserPrincipalName"],
    "Operation": "New-InboxRule",
    "ClientIP": compromised["IPAddress"],
    "ForwardTo": "collector@evil-demo.example",
    "AttackType": "inbox_forwarding",
})

print(len(inbox_attacks))
inbox_attacks.head()

In [ ]:
normal_rule_users = employees.iloc[rng.choice(len(employees), size=40, replace=False)]

inbox_normal = pd.DataFrame({
    "TimeGenerated": pd.Timestamp("2026-09-01", tz="UTC") + pd.to_timedelta(
        rng.integers(0, 30 * 86400, size=40), unit="s"
    ),
    "UserId": normal_rule_users["UserPrincipalName"].values,
    "Operation": "New-InboxRule",
    "ClientIP": normal_rule_users["HomeCity"].map(office_ip).values,
    "ForwardTo": rng.choice(["", "team@nordlys-demo.example"], size=40, p=[0.75, 0.25]),
    "AttackType": "normal",
})

print(len(inbox_normal))
inbox_normal.head()

In [ ]:
vpn_base = signins[
    (signins["ResultType"] == 0) & (signins["HomeCountry"] == "NO")
].sample(n=6, random_state=21)

vpn_normal = vpn_base[
    ["UserPrincipalName", "HomeCity", "HomeCountry", "TimeGenerated", "AppDisplayName"]
].copy()

vpn_normal["TimeGenerated"] = vpn_normal["TimeGenerated"] + pd.Timedelta(minutes=20)
vpn_normal["IPAddress"] = "203.0.113.250"
vpn_normal["Location"] = "DE"
vpn_normal["City"] = "Frankfurt"
vpn_normal["ResultType"] = 0
vpn_normal["AttackType"] = "normal"

print(len(vpn_normal))
vpn_normal.head()

In [ ]:
signins["AttackType"] = "normal"

all_signins = pd.concat(
    [signins, spray, travel, mfa, vpn_normal],
    ignore_index=True,
)
all_signins = all_signins.sort_values("TimeGenerated").reset_index(drop=True)

all_inbox = pd.concat(
    [inbox_normal, inbox_attacks],
    ignore_index=True,
)
all_inbox = all_inbox.sort_values("TimeGenerated").reset_index(drop=True)

print(len(all_signins))
print(len(all_inbox))

In [ ]:
from pathlib import Path

Path("../data").mkdir(exist_ok=True)
Path("../answer_key").mkdir(exist_ok=True)

signin_columns_to_drop = ["AttackType", "HomeCity", "HomeCountry"]

all_signins.drop(columns=signin_columns_to_drop).to_csv("../data/signin_logs.csv", index=False)
all_inbox.drop(columns=["AttackType"]).to_csv( "../data/inbox_events.csv", index=False)

all_signins[all_signins["AttackType"] != "normal"].to_csv("../answer_key/signin_truth.csv", index=False)
all_inbox[all_inbox["AttackType"] != "normal"].to_csv("../answer_key/inbox_truth.csv", index=False)

print("saved")